# Continuum: fine-tune a memory reranker

Fine-tunes `cross-encoder/ms-marco-MiniLM-L6-v2` (Apache 2.0, 23M parameters) on the
question → memory pairs from the data-generation notebook, then measures it on LongMemEval
exactly like the benchmark notebook: bge-small finds the top 20 turns, the reranker reorders them.

**How to run on Kaggle**
1. **Add Input** → your dataset made from the data notebook's outputs
   (`memory_retrieval_train_shard*.jsonl`, `memory_retrieval_validation_shard*.jsonl`).
2. Settings → Accelerator: **GPU T4** (one is enough). Settings → Internet: **On**.
3. Run All. Experiment as much as you like, judging only the **dev** results.
4. For the final model only, set `EVALUATE_TEST = True` and run once more. The test
   questions are never used for decisions, so their numbers stay honest.

**Output:** the model folder `continuum-memory-reranker/` and `training_report.json`
in `/kaggle/working`.

In [ ]:
import gc
import glob
import json
import math
import os
import random
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from huggingface_hub import hf_hub_download
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

SMOKE_TEST = os.environ.get("RERANKER_SMOKE_TEST") == "1"  # tiny local check

BASE_MODEL = "cross-encoder/ms-marco-MiniLM-L6-v2"
OUTPUT_MODEL_NAME = "continuum-memory-reranker"
TRAINING_FILES = "/kaggle/input/**/memory_retrieval_train_shard*.jsonl"
VALIDATION_FILES = "/kaggle/input/**/memory_retrieval_validation_shard*.jsonl"

GROUP_NEGATIVES = 7  # wrong lines per question in each training group
EPOCHS = 1
BATCH_SIZE = 16  # questions per step; each brings 1 answer + GROUP_NEGATIVES wrong lines
LEARNING_RATE = 2e-5
WARMUP_FRACTION = 0.1
TRAIN_MAX_LENGTH = 256  # tokens per (question, line) pair while training
EVAL_MAX_LENGTH = 512  # same as the benchmark notebook's rerankers
SEED = 0

LONGMEMEVAL_REPO = "xiaowu0162/longmemeval-cleaned"
LONGMEMEVAL_FILE = "longmemeval_s_cleaned.json"
LONGMEMEVAL_LIMIT = None  # use every question
FIRST_STAGE_MODEL = "BAAI/bge-small-en-v1.5"  # Continuum's retriever
FIRST_STAGE_QUERY_PREFIX = "Represent this sentence for searching relevant passages: "
RERANK_CANDIDATES = 20
DEV_QUESTIONS = 100  # fixed random subset for experimenting; the rest is the test set
SPLIT_SEED = 0
EVALUATE_TEST = False  # True only for the final model
RECALL_CUTOFFS = (1, 5, 10)
SESSION_CUTOFF = 5

if SMOKE_TEST:
    BASE_MODEL = "BAAI/bge-reranker-base"  # stand-in that is already in the local cache
    TRAINING_FILES = "memory_retrieval_train_shard*.jsonl"
    VALIDATION_FILES = "memory_retrieval_validation_shard*.jsonl"
    GROUP_NEGATIVES = 3
    BATCH_SIZE = 2
    LONGMEMEVAL_FILE = "longmemeval_oracle.json"
    LONGMEMEVAL_LIMIT = 60
    DEV_QUESTIONS = 20
    EVALUATE_TEST = True

OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print(f"Device: {DEVICE}" + (f" ({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else ""))

## Load the training data

In [ ]:
def read_jsonl(pattern):
    paths = sorted(glob.glob(pattern, recursive=True))
    rows = []
    for path in paths:
        with open(path, encoding="utf-8") as file:
            rows.extend(json.loads(line) for line in file if line.strip())
    return rows, paths


train, train_paths = read_jsonl(TRAINING_FILES)
validation, validation_paths = read_jsonl(VALIDATION_FILES)
if not train:
    raise SystemExit(f"No training files match {TRAINING_FILES}. Add the data notebook's output as an input.")

print(f"Training files: {[Path(p).name for p in train_paths]}")
print(f"{len(train):,} training and {len(validation):,} validation examples")
for source in sorted({row["source"] for row in train}):
    print(f"  {source:14s} {sum(row['source'] == source for row in train):7,}")
print(f"Hard negatives per example: {np.mean([len(row['negatives']) for row in train]):.1f} on average")

## Scoring helpers
A reranker scores each (question, line) pair. **Validation MRR** asks, for each held-out
generated question, how high its answer ranks among its own wrong lines: a quick signal
during training. LongMemEval below is the real test.

In [ ]:
def score_pairs(model, tokenizer, queries, passages, max_length, batch_size=128):
    """Raw relevance scores for (query, passage) pairs; higher means more relevant."""
    model.eval()
    scores = []
    with torch.inference_mode():
        for start in range(0, len(queries), batch_size):
            inputs = tokenizer(
                queries[start:start + batch_size], passages[start:start + batch_size],
                truncation=True, max_length=max_length, padding=True, return_tensors="pt",
            ).to(DEVICE)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEVICE == "cuda"):
                logits = model(**inputs).logits
            scores.extend(logits[:, 0].float().cpu().tolist())
    return np.asarray(scores, dtype=np.float32)


def validation_mrr(model, tokenizer, rows):
    if not rows:
        return float("nan")
    queries, passages, sizes = [], [], []
    for row in rows:
        group = [row["positive"], *row["negatives"]]
        queries += [row["query"]] * len(group)
        passages += group
        sizes.append(len(group))
    scores = score_pairs(model, tokenizer, queries, passages, TRAIN_MAX_LENGTH)
    ranks = [1 + int((group[1:] > group[0]).sum()) for group in np.split(scores, np.cumsum(sizes)[:-1])]
    return float(np.mean([1 / rank for rank in ranks]))

## Fine-tune
Each training group is a question, its answer and `GROUP_NEGATIVES` wrong lines. The loss
(softmax cross-entropy over the group) pushes the answer's score above every wrong line's.
Examples with too few hard negatives are topped up with answers from other conversations.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL).to(DEVICE)
rng = random.Random(SEED)


def training_group(example):
    negatives = [n for n in dict.fromkeys(example["negatives"]) if n != example["positive"]][:GROUP_NEGATIVES]
    for _ in range(10 * GROUP_NEGATIVES):
        if len(negatives) == GROUP_NEGATIVES:
            break
        other = rng.choice(train)
        if other["conversation_id"] != example["conversation_id"] and other["positive"] not in negatives:
            negatives.append(other["positive"])
    return [example["positive"], *negatives]


history = [{"epoch": 0, "validation_mrr": validation_mrr(model, tokenizer, validation)}]
print(f"Validation MRR before training: {history[0]['validation_mrr']:.3f}")

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
total_steps = EPOCHS * math.ceil(len(train) / BATCH_SIZE)
scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP_FRACTION * total_steps), total_steps)
scaler = torch.amp.GradScaler("cuda", enabled=DEVICE == "cuda")
started = time.perf_counter()

for epoch in range(1, EPOCHS + 1):
    model.train()
    order = list(range(len(train)))
    rng.shuffle(order)
    losses = []

    for start in tqdm(range(0, len(order), BATCH_SIZE), desc=f"epoch {epoch}"):
        batch = [train[i] for i in order[start:start + BATCH_SIZE]]
        groups = [training_group(example) for example in batch]
        queries = [example["query"] for example, group in zip(batch, groups) for _ in group]
        passages = [passage for group in groups for passage in group]

        inputs = tokenizer(queries, passages, truncation=True, max_length=TRAIN_MAX_LENGTH, padding=True, return_tensors="pt").to(DEVICE)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEVICE == "cuda"):
            logits = model(**inputs).logits[:, 0]

        # In every group the answer sits at index 0.
        group_logits = torch.split(logits.float(), [len(group) for group in groups])
        loss = torch.stack([
            F.cross_entropy(scores.unsqueeze(0), torch.zeros(1, dtype=torch.long, device=DEVICE))
            for scores in group_logits
        ]).mean()

        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        losses.append(loss.item())

    history.append({
        "epoch": epoch,
        "loss": float(np.mean(losses)),
        "validation_mrr": validation_mrr(model, tokenizer, validation),
    })
    print(f"Epoch {epoch}: loss {history[-1]['loss']:.4f}, validation MRR {history[-1]['validation_mrr']:.3f}")

training_minutes = (time.perf_counter() - started) / 60
del optimizer, scheduler, scaler  # frees GPU memory before evaluation loads a second model
gc.collect()
torch.cuda.empty_cache()

model_dir = OUTPUT_DIR / OUTPUT_MODEL_NAME
model.save_pretrained(model_dir)
tokenizer.save_pretrained(model_dir)
print(f"Trained in {training_minutes:.1f} min; saved to {model_dir}")

## Build the LongMemEval evaluation
Same setup as the benchmark notebook: every chat turn is a memory, each question searches its
own history, and bge-small picks the 20 candidates the rerankers reorder. Questions are split
once, with a fixed seed, into dev and test.

In [ ]:
instances = json.loads(Path(hf_hub_download(LONGMEMEVAL_REPO, LONGMEMEVAL_FILE, repo_type="dataset")).read_text(encoding="utf-8"))
if LONGMEMEVAL_LIMIT:
    instances = random.Random(SPLIT_SEED).sample(instances, min(LONGMEMEVAL_LIMIT, len(instances)))

question_ids = sorted(instance["question_id"] for instance in instances)
random.Random(SPLIT_SEED).shuffle(question_ids)
dev_ids = set(question_ids[:DEV_QUESTIONS])

texts, text_ids, questions = [], {}, []
for instance in instances:
    units, unit_sessions, relevant = [], [], set()
    for session_id, session in zip(instance["haystack_session_ids"], instance["haystack_sessions"]):
        for turn in session:
            content = turn["content"].strip()
            if not content:
                continue
            if content not in text_ids:
                text_ids[content] = len(texts)
                texts.append(content)
            if turn.get("has_answer"):
                relevant.add(len(units))
            units.append(text_ids[content])
            unit_sessions.append(session_id)
    questions.append({
        "id": instance["question_id"],
        "type": instance["question_type"],
        "question": instance["question"],
        "abstention": instance["question_id"].endswith("_abs"),
        "units": np.array(units, dtype=np.int64),
        "sessions": unit_sessions,
        "relevant": relevant,
        "answer_sessions": set(instance["answer_session_ids"]),
        "split": "dev" if instance["question_id"] in dev_ids else "test",
    })

retriever = SentenceTransformer(FIRST_STAGE_MODEL, device=DEVICE)
retriever.max_seq_length = 256
document_vectors = retriever.encode(texts, batch_size=128, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)
query_vectors = retriever.encode(
    [FIRST_STAGE_QUERY_PREFIX + q["question"] for q in questions],
    batch_size=128, normalize_embeddings=True, convert_to_numpy=True,
)
for question, query_vector in zip(questions, query_vectors):
    scores = document_vectors[question["units"]] @ query_vector
    question["first_stage_order"] = np.argsort(-scores, kind="stable")
    question["first_stage_top"] = float(scores[question["first_stage_order"][0]])
del retriever, document_vectors
gc.collect()

for split in ("dev", "test"):
    in_split = [q for q in questions if q["split"] == split]
    print(f"{split}: {sum(not q['abstention'] and bool(q['relevant']) for q in in_split)} answerable + "
          f"{sum(q['abstention'] for q in in_split)} abstention questions")

## Metrics (same definitions as the benchmark notebook)

In [ ]:
def first_relevant_rank(order, relevant):
    for rank, position in enumerate(order, start=1):
        if int(position) in relevant:
            return rank
    return None


def recall_at_k(order, relevant, k):
    return len({int(p) for p in order[:k]} & relevant) / min(k, len(relevant))


def session_recall_at_k(order, question, k):
    top_sessions = []
    for position in order:
        session = question["sessions"][int(position)]
        if session not in top_sessions:
            top_sessions.append(session)
            if len(top_sessions) == k:
                break
    return len(set(top_sessions) & question["answer_sessions"]) / min(k, len(question["answer_sessions"]))


def auroc(positives, negatives):
    if not positives or not negatives:
        return float("nan")
    p = np.asarray(positives)[:, None]
    n = np.asarray(negatives)[None, :]
    return float((p > n).mean() + 0.5 * (p == n).mean())


def evaluate(split_questions, rank):
    """`rank(question)` returns (turn positions best-first, top score)."""
    answerable = [q for q in split_questions if not q["abstention"] and q["relevant"]]
    abstention = [q for q in split_questions if q["abstention"] and len(q["units"])]
    rows, top_answerable = [], []
    for question in answerable:
        order, top_score = rank(question)
        found = first_relevant_rank(order, question["relevant"])
        row = {"id": question["id"], "type": question["type"], "rr": 0.0 if found is None else 1 / found}
        for k in RECALL_CUTOFFS:
            row[f"R@{k}"] = recall_at_k(order, question["relevant"], k)
        row[f"session R@{SESSION_CUTOFF}"] = session_recall_at_k(order, question, SESSION_CUTOFF)
        rows.append(row)
        top_answerable.append(top_score)

    summary = {"MRR": float(np.mean([row["rr"] for row in rows]))}
    for key in [f"R@{k}" for k in RECALL_CUTOFFS] + [f"session R@{SESSION_CUTOFF}"]:
        summary[key] = float(np.mean([row[key] for row in rows]))
    summary["abstention AUROC"] = auroc(top_answerable, [rank(q)[1] for q in abstention])
    by_type = {t: float(np.mean([r["rr"] for r in rows if r["type"] == t])) for t in sorted({r["type"] for r in rows})}
    return summary, {row["id"]: row["rr"] for row in rows}, by_type


def first_stage_rank(question):
    return question["first_stage_order"], question["first_stage_top"]


def reranker_rank_function(model, tokenizer, split_questions):
    """Score every question's top candidates once, then rank from the cached scores."""
    queries, passages, spans = [], [], {}
    for question in split_questions:
        pool = question["first_stage_order"][:RERANK_CANDIDATES]
        spans[question["id"]] = (len(queries), len(queries) + len(pool))
        queries += [question["question"]] * len(pool)
        passages += [texts[question["units"][int(p)]] for p in pool]
    scores = score_pairs(model, tokenizer, queries, passages, EVAL_MAX_LENGTH)

    def rank(question):
        start, end = spans[question["id"]]
        pool = question["first_stage_order"][:RERANK_CANDIDATES]
        local_order = np.argsort(-scores[start:end], kind="stable")
        return pool[local_order], float(scores[start:end][local_order[0]])

    return rank


def format_table(headers, rows):
    widths = [max(len(str(row[i])) for row in [headers, *rows]) for i in range(len(headers))]
    render = lambda row: "  ".join(
        str(cell).ljust(width) if i == 0 else str(cell).rjust(width)
        for i, (cell, width) in enumerate(zip(row, widths))
    )
    return "\n".join([render(headers), "  ".join("-" * width for width in widths), *map(render, rows)])

## Compare: bge-small alone, + original reranker, + fine-tuned reranker
"vs original" counts questions where the fine-tuned model ranks the first relevant turn higher
(+) or lower (-) than the original MiniLM reranker, the model to beat.

In [ ]:
original_model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL).to(DEVICE)
original_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
splits_to_run = ["dev", "test"] if EVALUATE_TEST else ["dev"]
report_results = {}

for split in splits_to_run:
    split_questions = [q for q in questions if q["split"] == split]
    methods = {
        "bge-small": first_stage_rank,
        "bge-small + original MiniLM": reranker_rank_function(original_model, original_tokenizer, split_questions),
        "bge-small + fine-tuned": reranker_rank_function(model, tokenizer, split_questions),
    }
    results = {name: evaluate(split_questions, rank) for name, rank in methods.items()}
    original_rr = results["bge-small + original MiniLM"][1]

    columns = ["MRR", "R@1", "R@5", "R@10", f"session R@{SESSION_CUTOFF}", "abstention AUROC"]
    rows = []
    for name, (summary, rr, _) in results.items():
        if name == "bge-small + fine-tuned":
            versus = f"+{sum(rr[q] > original_rr[q] for q in rr)} / -{sum(rr[q] < original_rr[q] for q in rr)}"
        else:
            versus = "baseline" if name == "bge-small + original MiniLM" else "-"
        rows.append([name, *(f"{summary[c]:.3f}" for c in columns), versus])

    print(f"\n{split.upper()} ({len(original_rr)} answerable questions)")
    print(format_table(["method", *columns, "vs original"], rows))

    types = sorted({t for _, _, by_type in results.values() for t in by_type})
    print(f"\n{split.upper()}: MRR by question type")
    print(format_table(["method", *types], [[name, *(f"{by_type.get(t, float('nan')):.3f}" for t in types)] for name, (_, _, by_type) in results.items()]))

    report_results[split] = {name: {"summary": summary, "mrr_by_type": by_type} for name, (summary, _, by_type) in results.items()}

if not EVALUATE_TEST:
    print("\nTest set not evaluated. Set EVALUATE_TEST = True only for the final model.")

report = {
    "base_model": BASE_MODEL,
    "training_files": [Path(p).name for p in train_paths],
    "training_examples": len(train),
    "validation_examples": len(validation),
    "config": {
        "epochs": EPOCHS, "batch_size": BATCH_SIZE, "group_negatives": GROUP_NEGATIVES,
        "learning_rate": LEARNING_RATE, "train_max_length": TRAIN_MAX_LENGTH, "seed": SEED,
    },
    "history": history,
    "training_minutes": round(training_minutes, 1),
    "longmemeval": {"file": LONGMEMEVAL_FILE, "dev_questions": DEV_QUESTIONS, "split_seed": SPLIT_SEED, "results": report_results},
}
(OUTPUT_DIR / "training_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
print(f"\nSaved {OUTPUT_DIR / 'training_report.json'}")

## How to read this
- **Validation MRR** rising during training means the model is learning the generated data.
  It doesn't prove anything about real memories; LongMemEval does.
- **Dev results** are for decisions: epochs, learning rate, data mix. Change one thing at a time.
- The fine-tuned model is worth keeping only if it **clearly beats the original MiniLM on dev**:
  higher MRR and many more wins than losses. Then run the test set once and record the numbers.
- Next, outside this notebook: check it on Continuum's one-line memories (local harness) and on
  LoCoMo before publishing.